# Building a Full Agent Loop with Claude

This notebook walks step-by-step through every layer of an agentic system:

| Section | What you'll build |
|---|---|
| **1 — Setup** | Client, model, env |
| **2 — Tool Runner (beta)** | Auto-loop via `@beta_tool` decorator |
| **3 — Manual Agent Loop** | Explicit loop: call → tool → result → repeat |
| **4 — Error Handling** | Retry logic, tool errors, API errors |
| **5 — Session Management** | Conversation history + tool use across turns |
| **6 — Subagents** | Spawning specialised child agents via the Agent SDK |
| **7 — Explicit Context Passing** | Sharing structured context between agents |

---
## Section 1 — Setup

In [1]:
# Install dependencies (run once)
# !pip install anthropic python-dotenv claude-agent-sdk

In [2]:
import json
import time
import random
from typing import Any
from datetime import datetime

import anthropic
from anthropic import beta_tool
from dotenv import load_dotenv

load_dotenv()

client = anthropic.Anthropic()

# claude-opus-4-6 is the most capable model; use claude-haiku-4-5 to save cost during exploration
MODEL = "claude-opus-4-6"

print(f"SDK version : {anthropic.__version__}")
print(f"Model       : {MODEL}")

SDK version : 0.84.0
Model       : claude-opus-4-6


### Shared tool implementations

We define real Python functions first; schemas come later in each section.

In [ ]:
# ── Tool implementations ──────────────────────────────────────────────────────

def get_weather(city: str) -> str:
    """Simulated weather lookup."""
    fake_data = {
        "london": "12°C, overcast",
        "paris": "18°C, sunny",
        "tokyo": "22°C, humid",
        "new york": "15°C, partly cloudy",
    }
    return fake_data.get(city.lower(), f"No data for {city}")


def calculate(expression: str) -> str:
    """Safely evaluate a simple arithmetic expression."""
    allowed = set("0123456789+-*/()., ")
    if not set(expression) <= allowed:
        return "Error: expression contains disallowed characters"
    try:
        result = eval(expression, {"__builtins__": {}})  # noqa: S307
        return str(result)
    except Exception as exc:
        return f"Error: {exc}"


def get_current_time() -> str:
    """Return current UTC time as ISO-8601."""
    return datetime.utcnow().isoformat() + "Z"


# Simple dispatcher used by the manual loop
TOOL_REGISTRY: dict[str, Any] = {
    "get_weather": lambda inp: get_weather(inp["city"]),
    "calculate": lambda inp: calculate(inp["expression"]),
    "get_current_time": lambda inp: get_current_time(),
}

print("Tools registered:", list(TOOL_REGISTRY))

---
## Section 2 — Tool Runner (Beta)

The **tool runner** is the simplest way to do agentic tool use.  
Decorate your functions with `@beta_tool`, pass them to `client.beta.messages.tool_runner()`, and the SDK handles the entire *call → execute → feed-back* loop for you.

In [ ]:
# ── 2a. Define tools with @beta_tool ─────────────────────────────────────────

@beta_tool
def get_weather_tool(city: str) -> str:
    """Get the current weather for a city.

    Args:
        city: The city name, e.g. London or Tokyo.
    """
    return get_weather(city)


@beta_tool
def calculate_tool(expression: str) -> str:
    """Evaluate a simple arithmetic expression such as '2 + 3 * 4'.

    Args:
        expression: Arithmetic expression using +, -, *, /, ().
    """
    return calculate(expression)


@beta_tool
def get_current_time_tool() -> str:
    """Return the current UTC time as an ISO-8601 string."""
    return get_current_time()


print("beta_tool objects:", [get_weather_tool, calculate_tool, get_current_time_tool])

In [ ]:
# ── 2b. Run the tool-runner loop ──────────────────────────────────────────────
# Each `message` in the iterator is a BetaMessage from one iteration of the loop.
# Iteration stops automatically when Claude produces end_turn (no more tool calls).

runner = client.beta.messages.tool_runner(
    model=MODEL,
    max_tokens=1024,
    tools=[get_weather_tool, calculate_tool, get_current_time_tool],
    messages=[{
        "role": "user",
        "content": (
            "What's the weather in London and Paris? "
            "Also calculate (12 + 8) * 3 and tell me the current UTC time."
        ),
    }],
)

for i, message in enumerate(runner):
    print(f"\n── Iteration {i} ── stop_reason={message.stop_reason}")
    for block in message.content:
        if block.type == "text":
            print(block.text)
        elif block.type == "tool_use":
            print(f"  [tool_call] {block.name}({block.input})")
        elif block.type == "tool_result":
            print(f"  [tool_result] → {block.content}")

---
## Section 3 — Manual Agent Loop

Use a manual loop when you need:
- Custom logging or approval gates between steps
- Conditional tool execution
- Fine-grained control over how history grows

In [ ]:
# ── 3a. Tool schemas (raw JSON) ───────────────────────────────────────────────

TOOLS_SCHEMA = [
    {
        "name": "get_weather",
        "description": "Return current weather for a given city.",
        "input_schema": {
            "type": "object",
            "properties": {
                "city": {"type": "string", "description": "City name, e.g. Tokyo"}
            },
            "required": ["city"],
        },
    },
    {
        "name": "calculate",
        "description": "Evaluate a simple arithmetic expression and return the result.",
        "input_schema": {
            "type": "object",
            "properties": {
                "expression": {
                    "type": "string",
                    "description": "Arithmetic expression, e.g. '(3 + 5) * 2'",
                }
            },
            "required": ["expression"],
        },
    },
    {
        "name": "get_current_time",
        "description": "Return the current UTC time as ISO-8601.",
        "input_schema": {"type": "object", "properties": {}, "required": []},
    },
]

In [ ]:
# ── 3b. The manual agent loop ─────────────────────────────────────────────────

def run_agent(
    user_message: str,
    system: str | None = None,
    max_iterations: int = 10,
    verbose: bool = True,
) -> str:
    """
    Run a full agentic loop until Claude stops calling tools or the
    iteration limit is reached.  Returns the final text response.
    """
    messages: list[dict] = [{"role": "user", "content": user_message}]
    params: dict = {
        "model": MODEL,
        "max_tokens": 2048,
        "tools": TOOLS_SCHEMA,
        "messages": messages,
    }
    if system:
        params["system"] = system

    for iteration in range(max_iterations):
        response = client.messages.create(**params)

        if verbose:
            print(f"\n[iter {iteration}] stop_reason={response.stop_reason}")

        # ── Done: no more tool calls ──────────────────────────────────────────
        if response.stop_reason == "end_turn":
            text = next((b.text for b in response.content if b.type == "text"), "")
            if verbose:
                print(f"Final answer:\n{text}")
            return text

        # ── Server hit its internal limit; re-send to resume ──────────────────
        if response.stop_reason == "pause_turn":
            messages.append({"role": "assistant", "content": response.content})
            params["messages"] = messages
            continue

        # ── Tool use ──────────────────────────────────────────────────────────
        if response.stop_reason == "tool_use":
            # 1. Append the full assistant turn (preserves tool_use blocks)
            messages.append({"role": "assistant", "content": response.content})

            # 2. Execute every requested tool
            tool_results = []
            for block in response.content:
                if block.type != "tool_use":
                    continue

                if verbose:
                    print(f"  → calling {block.name}({block.input})")

                fn = TOOL_REGISTRY.get(block.name)
                if fn is None:
                    result = f"Error: unknown tool '{block.name}'"
                    is_error = True
                else:
                    result = fn(block.input)
                    is_error = False

                if verbose:
                    print(f"     result: {result}")

                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": result,
                    "is_error": is_error,
                })

            # 3. Feed all results back in a single user turn
            messages.append({"role": "user", "content": tool_results})
            params["messages"] = messages

    return "[agent] Max iterations reached without a final answer."


# ── Run it ────────────────────────────────────────────────────────────────────
run_agent(
    "What's the weather in Tokyo? Then calculate (100 / 4) + 17. "
    "Finally, tell me the UTC time and wrap up in one sentence."
)

---
## Section 4 — Error Handling

Three things can go wrong:
1. **Tool execution error** — handle inside the loop with `is_error: true`
2. **Rate limit / server error** — exponential back-off retry
3. **Max iterations** — surface a clear message to the caller

In [ ]:
# ── 4a. Tool that raises an exception ────────────────────────────────────────

def flaky_tool(city: str) -> str:
    """Simulates a tool that sometimes fails."""
    if random.random() < 0.5:   # 50 % failure rate for demo
        raise RuntimeError("External API timeout")
    return get_weather(city)


def safe_execute(tool_name: str, tool_input: dict) -> tuple[str, bool]:
    """
    Execute a tool safely.  Returns (result_str, is_error).
    Errors are returned as informative strings so Claude can adapt.
    """
    fn = TOOL_REGISTRY.get(tool_name)
    if fn is None:
        return f"Error: unknown tool '{tool_name}'", True
    try:
        return str(fn(tool_input)), False
    except Exception as exc:
        return f"Tool error: {exc}", True


# Test it
result, is_err = safe_execute("calculate", {"expression": "2 ** 10"})
print(f"result={result!r}  is_error={is_err}")

result, is_err = safe_execute("unknown_tool", {})
print(f"result={result!r}  is_error={is_err}")

In [ ]:
# ── 4b. API retry with exponential back-off ───────────────────────────────────
# The SDK already retries on 429 and 5xx (max_retries=2 by default).
# This pattern adds custom retry logic for cases where you need more control.

def create_with_retry(
    max_attempts: int = 5,
    base_delay: float = 1.0,
    **kwargs,
) -> anthropic.types.Message:
    """
    Call client.messages.create with exponential back-off.
    Retries on rate-limit (429) and server errors (5xx).
    Raises immediately for client errors (4xx except 429).
    """
    last_exc: Exception | None = None
    for attempt in range(max_attempts):
        try:
            return client.messages.create(**kwargs)
        except anthropic.RateLimitError as exc:
            last_exc = exc
            retry_after = int(exc.response.headers.get("retry-after", "60"))
            print(f"[retry {attempt+1}] rate-limited — waiting {retry_after}s")
            time.sleep(retry_after)
        except anthropic.APIStatusError as exc:
            if exc.status_code >= 500:
                last_exc = exc
                delay = min(base_delay * (2 ** attempt) + random.uniform(0, 1), 60)
                print(f"[retry {attempt+1}] server error {exc.status_code} — waiting {delay:.1f}s")
                time.sleep(delay)
            else:
                raise  # 4xx (except 429) — do not retry
        except anthropic.APIConnectionError as exc:
            last_exc = exc
            delay = min(base_delay * (2 ** attempt), 30)
            print(f"[retry {attempt+1}] network error — waiting {delay:.1f}s")
            time.sleep(delay)

    raise last_exc  # type: ignore[misc]


print("create_with_retry defined — wraps client.messages.create with back-off")

In [ ]:
# ── 4c. Robust agent loop using safe_execute + create_with_retry ───────────────

def run_robust_agent(user_message: str, max_iterations: int = 10) -> str:
    messages = [{"role": "user", "content": user_message}]
    create_params = dict(model=MODEL, max_tokens=1024, tools=TOOLS_SCHEMA)

    for _ in range(max_iterations):
        response = create_with_retry(**create_params, messages=messages)

        if response.stop_reason == "end_turn":
            return next((b.text for b in response.content if b.type == "text"), "")

        if response.stop_reason == "pause_turn":
            messages.append({"role": "assistant", "content": response.content})
            continue

        # tool_use
        messages.append({"role": "assistant", "content": response.content})
        tool_results = []
        for block in response.content:
            if block.type != "tool_use":
                continue
            result, is_error = safe_execute(block.name, block.input)
            tool_results.append({
                "type": "tool_result",
                "tool_use_id": block.id,
                "content": result,
                "is_error": is_error,
            })
        messages.append({"role": "user", "content": tool_results})

    return "[agent] max iterations reached"


answer = run_robust_agent("What is 999 * 777? Also what's the weather in New York?")
print(answer)

---
## Section 5 — Session Management

A **session** is a persistent conversation thread.  The Claude API is stateless, so *your* code is responsible for maintaining the message history.  The `SessionManager` class below:

- Stores the full message history across multiple `chat()` calls
- Supports tool use inside each turn
- Exposes `history` so you can inspect or save a session
- Allows resuming from a saved session ID

In [ ]:
# ── 5a. SessionManager ────────────────────────────────────────────────────────

class SessionManager:
    """
    Manages a multi-turn conversation with optional tool use.
    Each call to `chat()` appends to the running history.
    """

    def __init__(
        self,
        session_id: str | None = None,
        system: str | None = None,
        tools: list[dict] | None = None,
    ):
        self.session_id: str = session_id or f"session-{int(time.time())}"
        self.system = system
        self.tools = tools or []
        self.history: list[dict] = []   # the full message log
        self._sessions: dict[str, list[dict]] = {}  # in-memory session store

    # ── public API ────────────────────────────────────────────────────────────

    def chat(self, user_message: str, verbose: bool = False) -> str:
        """Send a user message and return Claude's final text reply."""
        self.history.append({"role": "user", "content": user_message})

        params: dict = dict(
            model=MODEL,
            max_tokens=1024,
            messages=self.history,
        )
        if self.system:
            params["system"] = self.system
        if self.tools:
            params["tools"] = self.tools

        # inner agentic loop for this single turn
        for _ in range(10):
            response = client.messages.create(**params)

            if response.stop_reason == "end_turn":
                text = next((b.text for b in response.content if b.type == "text"), "")
                self.history.append({"role": "assistant", "content": text})
                return text

            if response.stop_reason == "pause_turn":
                self.history.append({"role": "assistant", "content": response.content})
                params["messages"] = self.history
                continue

            # tool_use
            self.history.append({"role": "assistant", "content": response.content})
            tool_results = []
            for block in response.content:
                if block.type != "tool_use":
                    continue
                if verbose:
                    print(f"  [tool] {block.name}({block.input})")
                result, is_error = safe_execute(block.name, block.input)
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": result,
                    "is_error": is_error,
                })
            self.history.append({"role": "user", "content": tool_results})
            params["messages"] = self.history

        return "[session] inner loop limit reached"

    def save(self) -> None:
        """Persist the current history under this session's ID."""
        self._sessions[self.session_id] = list(self.history)
        print(f"[session] saved '{self.session_id}' ({len(self.history)} messages)")

    @classmethod
    def resume(cls, session_id: str, instance: "SessionManager") -> "SessionManager":
        """
        Resume from a previously saved session.
        In production you would load from a database; here we use the in-memory store.
        """
        if session_id not in instance._sessions:
            raise KeyError(f"Session '{session_id}' not found")
        new_session = cls(
            session_id=session_id,
            system=instance.system,
            tools=instance.tools,
        )
        new_session.history = list(instance._sessions[session_id])
        new_session._sessions = instance._sessions
        print(f"[session] resumed '{session_id}' ({len(new_session.history)} messages)")
        return new_session

    def __repr__(self) -> str:
        return f"SessionManager(id={self.session_id!r}, turns={len(self.history)})"

In [ ]:
# ── 5b. Multi-turn session demo ───────────────────────────────────────────────

session = SessionManager(
    system="You are a helpful assistant with access to weather and calculator tools.",
    tools=TOOLS_SCHEMA,
)

# Turn 1
r1 = session.chat("What is 42 * 42?", verbose=True)
print("Turn 1:", r1)

# Turn 2 — Claude remembers this is a continuation
r2 = session.chat("Now subtract 100 from that result.", verbose=True)
print("Turn 2:", r2)

# Turn 3 — mix tool use with context from previous turns
r3 = session.chat("What's the weather in London and is it warm enough to go running?", verbose=True)
print("Turn 3:", r3)

print("\nSession state:", session)

In [ ]:
# ── 5c. Save and resume ───────────────────────────────────────────────────────

session.save()

# Simulate resuming in a new process / notebook restart
resumed = SessionManager.resume(session.session_id, session)

r4 = resumed.chat("Summarise our conversation so far in one sentence.")
print("After resume:", r4)

---
## Section 6 — Subagents

The **Claude Agent SDK** (`claude-agent-sdk`) lets you define named subagents with their own tools, system prompts, and permissions.  The parent agent can spawn them using the built-in `Agent` tool.

Architecture here:
```
Orchestrator agent
  ├── researcher  (WebSearch, Read)
  └── analyst     (calculate + custom tool)
```

In [ ]:
import anyio
from claude_agent_sdk import (
    query,
    ClaudeAgentOptions,
    AgentDefinition,
    ResultMessage,
    SystemMessage,
    AssistantMessage,
    TextBlock as SDKTextBlock,
)

print("claude_agent_sdk imported successfully")

In [ ]:
# ── 6a. Define subagents ──────────────────────────────────────────────────────

SUBAGENTS = {
    "researcher": AgentDefinition(
        description=(
            "A research specialist. Give it a question and it will search the web "
            "and return a concise factual summary."
        ),
        prompt=(
            "You are a precise research assistant. "
            "Search for information, synthesise the key facts, "
            "and return a short bullet-point summary. "
            "Always cite your sources."
        ),
        tools=["WebSearch", "WebFetch"],
    ),
    "analyst": AgentDefinition(
        description=(
            "A data analyst. Give it numbers or a description of a dataset "
            "and it will compute statistics and return insights."
        ),
        prompt=(
            "You are a quantitative analyst. "
            "Given data or numbers, compute relevant statistics and explain the results "
            "in plain English."
        ),
        tools=["Bash"],   # can run Python via bash for numeric work
    ),
}

print("Subagents defined:", list(SUBAGENTS))

In [ ]:
# ── 6b. Run orchestrator with subagents ───────────────────────────────────────

async def run_with_subagents(prompt: str) -> str:
    """
    Run the orchestrator.  It can delegate tasks to the researcher or analyst
    subagents by invoking the built-in Agent tool.
    """
    result_text = ""
    session_id = None

    async for message in query(
        prompt=prompt,
        options=ClaudeAgentOptions(
            allowed_tools=["Agent", "Read"],   # Agent = can spawn subagents
            agents=SUBAGENTS,
            max_turns=15,
            system_prompt=(
                "You are an orchestrator. You have two specialist subagents:\n"
                "- 'researcher': use for web research tasks\n"
                "- 'analyst': use for numerical / statistical tasks\n"
                "Delegate appropriately and synthesise their outputs into a final answer."
            ),
        ),
    ):
        # Capture the session ID from the init message
        if isinstance(message, SystemMessage) and message.subtype == "init":
            session_id = message.data.get("session_id")
            print(f"[session_id] {session_id}")

        elif isinstance(message, ResultMessage):
            result_text = message.result

    return result_text


# Run it
answer = anyio.run(
    run_with_subagents,
    "What is the current population of France? Then calculate what 0.01% of that would be.",
)
print("\n=== Final Answer ===")
print(answer)

---
## Section 7 — Explicit Context Passing

Subagents are stateless by default — each invocation is a fresh context.  When agents need to share data, you have two patterns:

| Pattern | How | When to use |
|---|---|---|
| **Inline context** | Include data directly in the prompt string | Small context (< 1 page) |
| **Structured context object** | Serialise a dict to JSON and embed it | Structured data, large context |

In [ ]:
# ── 7a. Inline context passing ────────────────────────────────────────────────

def pass_context_inline(previous_result: str, next_task: str) -> str:
    """
    Inject the previous result as explicit context in the next prompt.
    Useful for chaining two agents without sharing live session state.
    """
    prompt = (
        f"Here is the result from the previous step:\n"
        f"```\n{previous_result}\n```\n\n"
        f"Your task: {next_task}"
    )
    response = client.messages.create(
        model=MODEL,
        max_tokens=512,
        messages=[{"role": "user", "content": prompt}],
    )
    return next((b.text for b in response.content if b.type == "text"), "")


step1 = run_robust_agent("What is 123 * 456?")
print("Step 1 result:", step1)

step2 = pass_context_inline(step1, "Now express that number in scientific notation.")
print("Step 2 result:", step2)

In [ ]:
# ── 7b. Structured context object ─────────────────────────────────────────────

def build_context_object(
    task_description: str,
    prior_findings: dict,
    constraints: list[str] | None = None,
) -> dict:
    """Build a structured context dict that can be passed between agents."""
    return {
        "task": task_description,
        "timestamp": get_current_time(),
        "prior_findings": prior_findings,
        "constraints": constraints or [],
    }


def run_agent_with_context(context: dict) -> str:
    """
    Embed the structured context object into the system prompt so the agent
    has full awareness of the broader task and previous findings.
    """
    system = (
        "You are a specialist agent.  "
        "Your inputs, constraints, and prior work are provided as JSON below.\n\n"
        f"{json.dumps(context, indent=2)}"
    )
    response = client.messages.create(
        model=MODEL,
        max_tokens=512,
        system=system,
        messages=[{"role": "user", "content": context["task"]}],
    )
    return next((b.text for b in response.content if b.type == "text"), "")


# Build the context
ctx = build_context_object(
    task_description="Write a one-paragraph executive summary of the weather findings.",
    prior_findings={
        "london_weather": get_weather("london"),
        "paris_weather": get_weather("paris"),
        "tokyo_weather": get_weather("tokyo"),
    },
    constraints=["Tone: formal", "Max length: 100 words", "Mention all three cities"],
)

print("Context object:")
print(json.dumps(ctx, indent=2))

summary = run_agent_with_context(ctx)
print("\nAgent output:")
print(summary)

In [ ]:
# ── 7c. Full pipeline: gather → analyse → summarise ───────────────────────────

def run_pipeline(cities: list[str]) -> str:
    """
    3-stage pipeline with explicit context passing:
    1. Gather: collect weather data for each city
    2. Analyse: find patterns (uses tools)
    3. Summarise: write an executive summary (uses structured context)
    """
    # Stage 1 — data gathering (no LLM needed, pure tool calls)
    weather_data = {city: get_weather(city) for city in cities}
    print("[Stage 1] Gathered:", weather_data)

    # Stage 2 — analysis (agent with context + tools)
    analysis_ctx = build_context_object(
        task_description=(
            "Analyse the weather data for the given cities.  "
            "Calculate the temperature range and identify which city is warmest/coldest."
        ),
        prior_findings={"raw_weather": weather_data},
        constraints=["Focus on temperatures only"],
    )
    analysis = run_agent_with_context(analysis_ctx)
    print("\n[Stage 2] Analysis complete")

    # Stage 3 — summarise (inline context passing)
    summary = pass_context_inline(
        previous_result=analysis,
        next_task=(
            "Write a two-sentence travel advisory based on the weather analysis. "
            "Recommend the best city to visit today."
        ),
    )
    return summary


final = run_pipeline(["London", "Paris", "Tokyo", "New York"])
print("\n=== Travel Advisory ===")
print(final)

---
## Summary

| Concept | Key takeaway |
|---|---|
| **Tool Runner** | Simplest path — `@beta_tool` + `tool_runner()` handles the loop automatically |
| **Manual Loop** | Check `stop_reason`; append full `response.content`; feed all `tool_result`s in one user turn |
| **Error Handling** | Return `is_error: true` for tool failures; use exponential back-off for API errors |
| **Session Management** | Maintain `history` list; append both user and assistant turns including tool blocks |
| **Subagents** | Use `AgentDefinition` + `allowed_tools=["Agent"]`; capture `session_id` from init message |
| **Context Passing** | Inline: embed previous output in prompt string.  Structured: serialise a context dict to JSON in the system prompt |